# Semantic caching

**Track:** Portfolio. **Level:** Practitioner. **CPU:** yes. **Keys:** no.

## Goals

- Cache language-model answers under an exact key and under a similarity key.
- Measure hit rate, false-hit rate, and a latency stand-in on one threshold sweep.
- Drop cache entries when the corpus version changes.

## Sources and license

Original notebook. The design follows GPTCache (Zilliz, Apache-2.0): embed the question, return a stored answer when cosine similarity clears a threshold, otherwise call the model and store the result. The GPTCache source is not copied. The embedder is TF-IDF so the lab has no download and no key. A deployed cache would use the same encoder as the retriever.

## Concepts

An exact cache misses every paraphrase. A semantic cache trades those misses for a new error, the false hit, where a nearby but different question receives the wrong answer. Latency and token cost only move when the hit is true. The threshold is that tradeoff. A second axis is staleness: an answer cached under corpus version 1 is wrong after version 2, even if the question matches.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'pandas', 'scikit-learn', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from labs._shared.seed import seed_everything

seed_everything(0)


## Guided exercise: clusters, exact cache, semantic cache

Each cluster has tight paraphrases that share content words, plus one loose paraphrase that a bag-of-words vector is not expected to catch. The model stand-in burns CPU on purpose so a hit is measurably cheaper than a miss. Times are real. They are not a hosted API bill. The cost column is a fixed stand-in for token price: 0 on a hit, 1 on a miss.


In [ ]:
CLUSTERS = {
    "savings": {
        "answer": "The savings account yields 3 percent annual interest.",
        "tight": [
            "what is the savings account interest rate",
            "savings account interest rate",
            "interest rate of the savings account",
        ],
        "loose": "what yield do deposits pay",
    },
    "card": {
        "answer": "The credit card annual fee is 49 dollars.",
        "tight": [
            "what is the credit card annual fee",
            "credit card annual fee",
            "annual fee of the credit card",
        ],
        "loose": "how much does the plastic cost each year",
    },
}

vectorizer = TfidfVectorizer()
all_tight = [q for payload in CLUSTERS.values() for q in payload["tight"]]
vectorizer.fit(all_tight)

def embed(texts: list[str]) -> np.ndarray:
    return vectorizer.transform(texts).toarray()

def model_call(cluster: str) -> tuple[str, float]:
    started = time.perf_counter()
    acc = 0
    for i in range(4000):
        acc ^= (hash(cluster) + i) & 0xFFFF
    elapsed = time.perf_counter() - started
    return CLUSTERS[cluster]["answer"] + f" [{acc & 0}]", elapsed

class ExactCache:
    def __init__(self):
        self.store = {}

    def get(self, key: str):
        return self.store.get(key)

    def put(self, key: str, value: str) -> None:
        self.store[key] = value

class SemanticCache:
    def __init__(self, threshold: float, version: int = 1):
        self.threshold = threshold
        self.version = version
        self.entries = []

    def get(self, query: str, version: int):
        if not self.entries:
            return None
        q = embed([query])[0]
        best, best_sim = None, -1.0
        for entry in self.entries:
            if entry["version"] != version:
                continue
            sim = float(q @ entry["vector"])
            if sim > best_sim:
                best, best_sim = entry, sim
        if best is not None and best_sim >= self.threshold:
            return {"answer": best["answer"], "cluster": best["cluster"], "sim": best_sim}
        return None

    def put(self, query: str, cluster: str, answer: str, version: int) -> None:
        self.entries.append({
            "vector": embed([query])[0],
            "cluster": cluster,
            "answer": answer,
            "version": version,
        })

def run_cache(cache_mode: str, threshold: float = 0.55, version: int = 1):
    exact = ExactCache()
    semantic = SemanticCache(threshold, version=version)
    rows = []
    # Warm the cache with the first tight paraphrase of each cluster.
    for cluster, payload in CLUSTERS.items():
        answer, _ = model_call(cluster)
        exact.put(payload["tight"][0], answer)
        semantic.put(payload["tight"][0], cluster, answer, version)
    queries = []
    for cluster, payload in CLUSTERS.items():
        for text in payload["tight"][1:]:
            queries.append((cluster, text, True))
        queries.append((cluster, payload["loose"], False))
    for gold, text, expect_hit in queries:
        started = time.perf_counter()
        if cache_mode == "exact":
            hit = exact.get(text)
            if hit is None:
                answer, model_time = model_call(gold)
                exact.put(text, answer)
                latency = time.perf_counter() - started + model_time
                rows.append({"hit": False, "true_hit": False, "false_hit": False, "latency": latency, "cost": 1})
            else:
                rows.append({"hit": True, "true_hit": hit.startswith(CLUSTERS[gold]["answer"][:12]), "false_hit": False, "latency": time.perf_counter() - started, "cost": 0})
        else:
            found = semantic.get(text, version)
            if found is None:
                answer, model_time = model_call(gold)
                semantic.put(text, gold, answer, version)
                rows.append({"hit": False, "true_hit": False, "false_hit": False, "latency": time.perf_counter() - started + model_time, "cost": 1})
            else:
                true_hit = found["cluster"] == gold
                rows.append({
                    "hit": True,
                    "true_hit": true_hit,
                    "false_hit": not true_hit,
                    "latency": time.perf_counter() - started,
                    "cost": 0,
                    "expect_hit": expect_hit,
                })
    return pd.DataFrame(rows)

exact_df = run_cache("exact")
semantic_df = run_cache("semantic", threshold=0.55)
print("exact hit rate", round(exact_df["hit"].mean(), 3), "mean latency", round(exact_df["latency"].mean(), 4))
print("semantic hit rate", round(semantic_df["hit"].mean(), 3), "false-hit rate", round(semantic_df["false_hit"].mean(), 3))
assert exact_df["hit"].mean() == 0.0  # every test line is a paraphrase, not a duplicate
assert semantic_df["true_hit"].sum() >= 2
assert semantic_df.loc[semantic_df["hit"], "latency"].mean() < semantic_df.loc[~semantic_df["hit"], "latency"].mean()


## Your turn

Sweep the cosine threshold. Hit rate should fall as the threshold rises, and the false-hit rate should not rise. Plot both curves and the mean cost.

## Solution


In [ ]:
sweep = []
for threshold in np.round(np.linspace(0.2, 0.95, 8), 2):
    frame = run_cache("semantic", threshold=float(threshold))
    sweep.append({
        "threshold": float(threshold),
        "hit_rate": float(frame["hit"].mean()),
        "false_hit_rate": float(frame["false_hit"].mean()),
        "mean_cost": float(frame["cost"].mean()),
        "mean_latency": float(frame["latency"].mean()),
    })
sweep_df = pd.DataFrame(sweep)
print(sweep_df.round(3).to_string(index=False))
assert sweep_df["hit_rate"].iloc[0] >= sweep_df["hit_rate"].iloc[-1]
assert sweep_df["false_hit_rate"].max() <= sweep_df["false_hit_rate"].iloc[0] + 1e-9

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(sweep_df["threshold"], sweep_df["hit_rate"], marker="o", label="hit rate")
ax.plot(sweep_df["threshold"], sweep_df["false_hit_rate"], marker="o", label="false-hit rate")
ax.plot(sweep_df["threshold"], sweep_df["mean_cost"], marker="o", label="mean cost")
ax.set_xlabel("cosine threshold")
ax.legend()
ax.set_title("Semantic cache tradeoff")
plt.tight_layout()
plt.show()


## Beyond the course

Store an answer under corpus version 1. Query the same string under version 2. The cache must miss, otherwise it serves a stale fee after the catalog changes. This is the invalidation experiment.


In [ ]:
cache = SemanticCache(threshold=0.5, version=1)
cache.put(CLUSTERS["savings"]["tight"][0], "savings", CLUSTERS["savings"]["answer"], version=1)
stale = cache.get(CLUSTERS["savings"]["tight"][1], version=1)
fresh = cache.get(CLUSTERS["savings"]["tight"][1], version=2)
print("version 1", None if stale is None else stale["cluster"], "version 2", fresh)
assert stale is not None and fresh is None
print("LAB OK")


## Research questions

1. The loose paraphrases are in the query list. Which threshold, if any, caught them, and what encoder would you measure next?
2. Cost here is 0 or 1. How do you extend the column when a hit still spends tokens on a verifier that checks the cached answer against the new corpus?
3. Two users ask the same question and are allowed to see different account fees. What has to enter the cache key besides the embedding?

## Further reading

- Bang, *GPTCache* (Zilliz, Apache-2.0). Design reference, not copied. https://github.com/zilliztech/GPTCache
- The threshold curves above are the lab's own measurement on a TF-IDF encoder.
- See the production RAG lab for the encoder this cache should share with retrieval.
